# Remote Work and Wage Inequality: Correcting Bias in Regression with Generated Binary Labels

In [ ]:
# Run this cell first on Google Colab to install the bias-correction package.
# The explicit jax pin avoids a circular-import error from Colab's preinstalled jax.
%pip install "ValidMLInference>=1.4.0" "jax>=0.4.34"

This notebook estimates the association between working from home and salaries using real-world job postings data [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734). It illustrates how the function `ols_bcm` can be used to correct bias from regressing on AI/ML-generated labels. The notebook reproduces results from Table 1 of [Battaglia, Christensen, Hansen & Sacher (2024)](https://arxiv.org/abs/2402.15585).

## Data

The package contains a subset of [a larger dataset](https://wfhmap.com/) regarding work from home. The sample consists of 16,315 job postings for 2022 and 2023 with “San Diego, CA” recorded as the city and “72” recorded as the NAICS2 industry code of the advertising firm. 

The data set contains the following entries:
1. `city_name` 
2. `naics_2022_2` - an industry code 
3. `salary` 
4. `wfh_rwo` - ML-generated indicator of whether the job offers work from home using fine-tuned DistilBERT as in [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734)
5. `soc_2021_2` - Bureau of Labor Statistics Standard Occupational Classification code
6. `employment_type_name` - indicates whether the position is full-time or part-time 

In [ ]:
from ValidMLInference import remote_work_data, ols, ols_bcm
import numpy as np

SD_data = remote_work_data()
SD_data.head(5)

For purpose of this estimation, we also log-transform the salary data. 

In [ ]:
SD_data['salary'] = np.log(SD_data['salary'])

## Estimating the false-positive rate

The variable `wfh_rwo` describing whether the job posting offers remote work is not manually collected, but is imputed via ML methods using fine-tuned DistilBERT as in [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734). This classifier has over 99% test accuracy in the full sample. Nevertheless, as [Battaglia, Christensen, Hansen & Sacher (2024)](https://arxiv.org/abs/2402.15585) document, even high-performance classifiers can lead to large biases in OLS estimates.

The bias correction methods `ols_bca` and `ols_bcm` require estimates of the classifier's false-positive rate.

We estimate the false positive rate manually. To do so, we took a random sample of size 1000 postings. Of these, 26 had `wfh_rwo = 1`. Based on reading these 26 postings, 9 appeared to be misclassified. This means the estimated false-positive rate is 0.009. Accordingly, we will implement `ols_bcm` with `fpr = 0.009` (the estimated false-positive rate) and `m = 1000` (the sample size used to estimate the false-positive rate).

## Results

We first present results for a simple regression of log salary onto the remote work indicator. We then consider a second specification with fixed effects.

We compare standard OLS estimates and confidence intervals with the estimates and confidence intervals from `ols_bcm`, which performs a direct bias correction and computes bias-corrected CIs.

### Without fixed effects

We first present OLS estimates:

In [ ]:
res = ols(formula = "salary ~ wfh_rwo", data=SD_data, intercept = True)
print(res.summary())

Now using the multiplicative bias correction, with bias corrected CIs:

In [ ]:
res = ols_bcm(formula= "salary ~ wfh_rwo", data=SD_data, fpr = 0.009, m = 1000, intercept=True)
print(res.summary())

### With fixed effects

We repeat the above now with fixed effects, which are easily generated for the categorical variables `soc_2021_2` and `employment_type_name`.

First using OLS:

In [ ]:
res = ols(formula = "salary ~ wfh_rwo + C(soc_2021_2) + C(employment_type_name)", data = SD_data, intercept=True)
summary = res.summary()
rows = ["Intercept", "wfh_rwo"]
print(summary.loc[rows])

Now using the multiplicative bias correction:

In [ ]:
res = ols_bcm(formula = "salary ~ wfh_rwo + C(soc_2021_2) + C(employment_type_name)", generated_var = "wfh_rwo", data = SD_data, fpr = 0.009, m=1000)
summary = res.summary()
print(summary.loc[rows])

Comparing these results with the OLS results above, we see that the bias corrected CI for the slope coefficient lies to the right of the OLS point estimate.